<a href="https://colab.research.google.com/github/Sebastianelli-Nicola/Traffic-Driven-EV-Queuing-Predictor/blob/main/reconstruction_missing_period.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ricostruzione e Imputazione Avanzata di Serie Storiche di Traffico

Il presente notebook illustra il processo completo di imputazione e ricostruzione dei dati mancanti (*Data Imputation*) all'interno di serie storiche relative al traffico veicolare.

Si applica un algoritmo customizzato basato sulla ricerca di periodi analoghi (simil-KNN) e sull'iniezione di varianza locale, ottimizzato tramite Grid Search bidirezionale. L'obiettivo è colmare gap temporali estesi garantendo la coerenza statistica, la preservazione dei pattern orari pendolari e l'adattamento ai cali fisiologici legati alla stagionalità (es. periodo estivo).

Il processo culmina con una validazione avanzata che analizza le metriche di errore e la forma delle distribuzioni sia su scala Macro (mensile) che Micro (settimane adiacenti).

## 1. Importazione Librerie e Pre-elaborazione Dati
In questa prima fase si importano le librerie necessarie (Pandas, NumPy, Seaborn, Matplotlib, ecc.) e si effettua il montaggio dell'ambiente Google Drive per l'accesso ai file. Successivamente, si caricano i dataset storici (traffico e anagrafica stazioni). Si eseguono le operazioni di pulizia strutturale, il parsing rigoroso dei formati data/ora e la creazione della variabile target `differenza` (traffico libero vs attuale), assicurandosi di imporre il limite fisico dello zero.

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from google.colab import drive

# ==========================================
# 1. CARICAMENTO E PRE-ELABORAZIONE
# ==========================================
print("Montaggio Google Drive e caricamento dati...")
drive.mount('/content/drive')

path_traffico = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_originale/traffico.csv'
path_stazioni = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_originale/stazioni.csv'

df_tr = pd.read_csv(path_traffico)
df_st = pd.read_csv(path_stazioni)

df_tr.columns = df_tr.columns.str.strip()
df_st.columns = df_st.columns.str.strip()

df_tr['giorno'] = pd.to_datetime(df_tr['giorno'], errors='coerce')
df_tr['orario'] = df_tr['orario'].astype(str).str.strip()
df_tr['timestamp'] = pd.to_datetime(df_tr['giorno'].astype(str) + " " + df_tr['orario'], errors='coerce')

df_tr['velocita_attuale'] = pd.to_numeric(df_tr['velocita_attuale'], errors='coerce')
df_tr['velocita_senza_traffico'] = pd.to_numeric(df_tr['velocita_senza_traffico'], errors='coerce')

df_tr['differenza'] = df_tr['velocita_senza_traffico'] - df_tr['velocita_attuale']
df_tr['differenza'] = df_tr['differenza'].clip(lower=0)

df = df_tr.merge(df_st, left_on='id_stazione', right_on='id', how='left')
df = df.dropna(subset=['timestamp', 'differenza']).sort_values(['id_stazione', 'timestamp']).reset_index(drop=True)

df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.weekday